# inference npe

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pickle
import torch
import optuna
import arviz as az
from pathlib import Path
RESULTS_DIR = configure_run(PROJECT)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from scipy.stats import gaussian_kde
from scipy.special import rel_entr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform
import warnings
import sys
sys.path.insert(0, str(PROJECT / "analogues"))
from splits import pair_groups, group_split

optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# load pickles and build clean DataFrames
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)
    
keep_idx    = np.load(tng_dir / "keep_idx.npy")



tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_mass_sample = np.load(tng_dir / "stellar_mass.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_same_host   = np.load(tng_dir / "have_same_host.npy").astype(bool)
tng_grnr = np.load(tng_dir / "grnr.npy")

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

In [ ]:
BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio",
                        "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio",
                        "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}

TARGET = "log_m200c"

# we only use quantities which are calculated from position and velocities for inference.
# seperation 761 +- 11 kpc Li et al. 2021
# vr,vt van der Marel et al. 2012
# vt contested Salomon et al. 2021 give vt = 82.4 +-31.2 km/s. We run both
# log_mass_ratio: ratio of total bound masses not stellar. Sawala et al. 2023. M_mw = 1.1 +- 0.2
# M_31 = 1.5 +- 0.5 x 10^12 Msun.
# One thing to note though boundsubhalo/totalmass is not equal to observational m200 and does not cancel exactly in a ratio; this is a proxy assumption
LG_OBS, LG_UNC = lg_observation("vdM2012")



LG_SAFE_BLOCKS = ["core_dynamics"]

# We only use blocks which have defensible values and skip other blocks. Those are only used for simulation comparsions. 

VT_VARIANTS = {"vdM2012": 17.0, "Salomon2021": 82.4}

print("Real LG observation (inference-safe covariates only):")
for k in LG_OBS:
    print(k, LG_OBS[k], "uncertainty:", LG_UNC[k] if LG_UNC[k] is not None else "point scenario only")
print(f"LG-safe blocks: {LG_SAFE_BLOCKS}")

In [ ]:
# reuse pair_features functions from feature_analysis
def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]

    gr_i    = sample["g_r"][pi];          gr_j    = sample["g_r"][pj]
    sigma_i = sample["vel_disp"][pi];     sigma_j = sample["vel_disp"][pj]
    kap_i   = sample["kappa_rot"][pi];    kap_j   = sample["kappa_rot"][pj]
    met_i   = met_transform["flamingo_met_dex"][pi]
    met_j   = met_transform["flamingo_met_dex"][pj]

    df = pd.DataFrame({
        "gr_mean":        (gr_i    + gr_j)    / 2,
        "gr_diff":        np.abs(gr_i    - gr_j),
        "met_mean":       (met_i   + met_j)   / 2,
        "met_diff":       np.abs(met_i   - met_j),
        "sigma_mean":     (sigma_i + sigma_j) / 2,
        "sigma_diff":     np.abs(sigma_i - sigma_j),
        "kappa_mean":     (kap_i   + kap_j)   / 2,
        "kappa_diff":     np.abs(kap_i   - kap_j),
        "separation":     pairs["separation_kpc"],
        "vr":             pairs["vr"],
        "vt":             pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"],
        "log_m200c":      pairs["log_tot_virial_mass"],
        "host_i": sample["host_top_index"][pi],
        "host_j": sample["host_top_index"][pj],
    })
    return df

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):

    gr_i  = tng_gr_sample[tng_pairs_i];  gr_j  = tng_gr_sample[tng_pairs_j]

    # metallicity: log10(Z/Zsun) z-scored with TNG's own scaler
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]

    df = pd.DataFrame({
        "gr_mean":    (gr_i        + gr_j)        / 2,
        "gr_diff":    np.abs(gr_i        - gr_j),
        "met_mean":   (met_i       + met_j)        / 2,
        "met_diff":   np.abs(met_i       - met_j),
        "sigma_mean": (tng_sigma_i + tng_sigma_j)  / 2,
        "sigma_diff": np.abs(tng_sigma_i - tng_sigma_j),
        "kappa_mean": (tng_kappa_i + tng_kappa_j)  / 2,
        "kappa_diff": np.abs(tng_kappa_i - tng_kappa_j),
        "separation": tng_separation,
        "vr":         tng_vr,
        "vt":         tng_vt,
        "log_mass_ratio": tng_log_mass_ratio,
        "log_m200c":  tng_log_m200c,               # target for now will add some more later
        "host_i": tng_grnr[tng_pairs_i],
        "host_j": tng_grnr[tng_pairs_j],
    })
    return df

flam_df = pair_features_flamingo(sample, pairs, met_transform)
tng_df  = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)

# add same_host flag
flam_df["same_host"] = pairs["have_same_host"].astype(bool)
tng_df["same_host"]  = tng_same_host

# clean splits
ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])

# same/diff host subsets — FLAMINGO only (TNG too small to split further)
flam_same = flam_df_clean[flam_df_clean["same_host"]].copy()
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

print(f"FLAMINGO full:      {len(flam_df_clean):,}")
print(f"FLAMINGO same-host: {len(flam_same):,}")
print(f"FLAMINGO diff-host: {len(flam_diff):,}")
print(f"TNG full:           {len(tng_df_clean):,}")
print(f"TNG same-host:      {tng_df_clean['same_host'].sum():,}")
print(f"TNG diff-host:      {(~tng_df_clean['same_host']).sum():,}")

In [ ]:
for name, df in [("flam_full", flam_df_clean), ("flam_same", flam_same),
                 ("flam_diff", flam_diff), ("tng", tng_df_clean)]:
    g = pair_groups(df)
    sizes = np.bincount(g); sizes = sizes[sizes > 0]
    print(f"{name:<10} N={len(df):>7,}  components={len(sizes):>6,}  "
          f"largest={sizes.max():>6,} ({100*sizes.max()/len(df):5.1f}%)  "
          f"median={np.median(sizes):.0f}")

In [ ]:
# from sklearn.metrics import r2_score
# 
# def posterior_point_predictions(posterior, X_raw, scaler, device, n_samples=500):
#     X_sc = scaler.transform(X_raw.astype(np.float32))
#     x_batch = torch.tensor(X_sc, device=device)
#     with torch.no_grad():
#         samples = posterior.sample_batched((n_samples,), x=x_batch)
#     return samples.mean(dim=0).squeeze(-1).cpu().numpy()
# 
# baseline_dir = PROJECT / "pickles_v1"
# with open(baseline_dir / "flamingo_npe_checkpoint.pkl", "rb") as f:
#     v1 = pickle.load(f)
# v1_post, v1_scal = v1["posteriors"], v1["scalers"]
# 
# print(f"{'Block':<18} {'%contam':>8} {'R² all test':>12} {'R² clean only':>14} {'n_clean':>8}")
# for block_name, feats in BLOCKS.items():
#     cols  = feats + [TARGET, "host_i", "host_j"]
#     valid = flam_df_clean[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)
# 
#     tr, te = train_test_split(np.arange(len(valid)), test_size=0.15, random_state=42)
#     # confirm this reproduces the v1 partition
#     _, te_df = train_test_split(valid, test_size=0.15, random_state=42)
#     assert np.array_equal(np.sort(te), np.sort(te_df.index.values)), "split mismatch"
# 
#     g = pair_groups(valid)
#     contam = np.isin(g[te], np.unique(g[tr]))
# 
#     key   = f"{block_name}__full"
#     preds = posterior_point_predictions(v1_post[key], valid.iloc[te][feats].values,
#                                         v1_scal[key], device)
#     y = valid.iloc[te][TARGET].values
#     print(f"{block_name:<18} {100*contam.mean():>7.2f}% {r2_score(y, preds):>12.4f} "
#           f"{r2_score(y[~contam], preds[~contam]):>14.4f} {(~contam).sum():>8,}")

In [ ]:
from scipy.stats import ks_2samp
ks_stat, p = ks_2samp(flam_df_clean["log_m200c"], tng_df_clean["log_m200c"])
print(f"theta-marginal KS: {ks_stat:.4f}  (p={p:.3e})")

In [ ]:
flam_med = flam_df_clean["log_m200c"].median()
tng_med  = tng_df_clean["log_m200c"].median()
print(f"FLAMINGO median log M200c (training): {flam_med:.4f}")
print(f"TNG median log M200c (training):      {tng_med:.4f}")
print(f"Difference: {flam_med - tng_med:.4f} dex")

In [ ]:
# # optimizing on core_dynamics then apply best arch to all blocks
# FEATS_OPT = BLOCKS["core_dynamics"]
# prior = BoxUniform(
#     low=torch.tensor([11.5], device=device),
#     high=torch.tensor([14.5], device=device)
# )

# def make_tensors(df, feats, scaler=None, fit_scaler=False):
#     X = df[feats].values.astype(np.float32)
#     y = df[TARGET].values.astype(np.float32)
#     if fit_scaler:
#         scaler = StandardScaler()
#         X = scaler.fit_transform(X)
#     else:
#         X = scaler.transform(X)
#     return (torch.tensor(X, device=device), torch.tensor(y.reshape(-1, 1), device=device),
#             scaler)

# # train/val split on FLAMINGO for Optuna
# flam_train, flam_val = train_test_split(
#     flam_df_clean, test_size=0.2, random_state=42
# )

# X_tr, y_tr, scaler_opt = make_tensors(flam_train, FEATS_OPT, fit_scaler=True)
# X_val, y_val, _        = make_tensors(flam_val,   FEATS_OPT, scaler_opt)

# def optuna_objective(trial):
#     hidden_features = trial.suggest_categorical(
#         "hidden_features", [32, 64, 128, 256]
#     )
#     num_transforms  = trial.suggest_int("num_transforms", 2, 8)
#     learning_rate   = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
#     batch_size      = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
#     num_layers      = trial.suggest_int("num_layers", 1, 3)

#     try:
#         density_builder = posterior_nn(
#             model="maf",
#             hidden_features=hidden_features,
#             num_transforms=num_transforms,
#         )
#         inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
#         inference.append_simulations(y_tr, X_tr)
#         density_est = inference.train(
#             training_batch_size=batch_size,
#             learning_rate=learning_rate,
#             max_num_epochs=50,
#             stop_after_epochs=10,
#             show_train_summary=False,
#         )
#         with torch.no_grad():
#             log_probs = density_est.log_prob(y_val, condition=X_val)
#         return log_probs.mean().item()

#     except Exception:
#         return -1e10

# study = optuna.create_study(direction="maximize",
#                              sampler=optuna.samplers.TPESampler(seed=42))
# study.optimize(
#     optuna_objective, n_trials=30,
#     show_progress_bar=True,
#     n_jobs=1 if device == "cuda" else -1,
# )

# best_params = study.best_params
# print("\nBest hyperparameters:")
# for k, v in best_params.items():
#     print(f"  {k}: {v}")

In [ ]:
# import pickle
# from pathlib import Path

# CHECKPOINT_PATH = Path("flamingo_npe_checkpoint.pkl")
# TNG_CHECKPOINT_PATH = Path("tng_npe_checkpoint.pkl")

# stale_flam_keys = [f"{b}__{s}" for b in ["core_chemistry", "all_features"]

#                                 for s in ["full", "same_host", "diff_host"]]
# stale_tng_keys = ["core_chemistry", "all_features"]

# for path, stale_keys in [(CHECKPOINT_PATH, stale_flam_keys), (TNG_CHECKPOINT_PATH, stale_tng_keys)]:
#     if path.exists():
#         with open(path, "rb") as f:
#             ckpt = pickle.load(f)
#         for d in ["posteriors", "scalers", "samples", "test_scores"]:
#             if d in ckpt:
#                 for k in stale_keys:
#                     ckpt[d].pop(k, None)
#         with open(path, "wb") as f:
#             pickle.dump(ckpt, f)
#         print(f"{path}: purged {len(stale_keys)} stale keys")
#     else:
#         print(f"{path}: does not exist yet, nothing to purge")

In [ ]:
import pickle
import matplotlib.pyplot as plt

with open(LEGACY_RESULTS_DIR / "optuna_search_results.pkl", "rb") as f:
    result = pickle.load(f)
study = result["study"]
df = study.trials_dataframe()

plt.figure(figsize=(7,4))
plt.plot(df["number"], df["value"], "o-")
plt.xlabel("Trial")
plt.ylabel("Validation log-prob (higher = better)")
plt.title("Optuna search progress")
plt.show()
 # For training cell since rn I just load pickle file
best_params = result["best_params"]
print("Loaded best_params:", best_params)

prior = BoxUniform(
    low=torch.tensor([11.5], device=device),
    high=torch.tensor([14.5], device=device)
)

In [ ]:
df = study.trials_dataframe()
cols = ["number", "value", "duration",
         "params_hidden_features", "params_num_transforms",
         "params_batch_size", "params_learning_rate"]
df[cols].sort_values("value", ascending=False)

df.groupby("params_hidden_features")["value"].agg(["mean", "max", "count"])
df.groupby("params_num_transforms")["value"].agg(["mean", "max", "count"])

In [ ]:
import pickle
from pathlib import Path

def try_load(path, label):
    p = Path(path)
    if p.exists():
        with open(p, "rb") as f:
            data = pickle.load(f)
        print(f"  loaded {label}  ({path})")
        return data
    print(f"  MISSING {label}  ({path}) -- not found, skipping")
    return None

print("Loading previously computed results:")

optuna_result  = try_load(LEGACY_RESULTS_DIR / "optuna_search_results.pkl", "Optuna study + best_params")
best_params = optuna_result["best_params"] if optuna_result else None
study       = optuna_result["study"] if optuna_result else None


flam_ckpt      = try_load(RESULTS_DIR / "flamingo_npe_checkpoint.pkl", "FLAMINGO posteriors/scalers/samples")
flam_posteriors  = flam_ckpt.get("posteriors", {})  if flam_ckpt else {}
flam_scalers     = flam_ckpt.get("scalers", {})     if flam_ckpt else {}
flam_samples     = flam_ckpt.get("samples", {})     if flam_ckpt else {}
flam_test_scores = flam_ckpt.get("test_scores", {}) if flam_ckpt else {}
if flam_ckpt and best_params is None:
    best_params = flam_ckpt.get("best_params")

tng_ckpt       = try_load(RESULTS_DIR / "tng_npe_checkpoint.pkl", "TNG posteriors/scalers/samples")
tng_posteriors  = tng_ckpt.get("posteriors", {})  if tng_ckpt else {}
tng_scalers     = tng_ckpt.get("scalers", {})     if tng_ckpt else {}
tng_samples     = tng_ckpt.get("samples", {})     if tng_ckpt else {}
tng_test_scores = tng_ckpt.get("test_scores", {}) if tng_ckpt else {}

js_results     = try_load(RESULTS_DIR / "js_divergence_results.pkl", "JS divergence table")
tng_r2_results = try_load(RESULTS_DIR / "tng_cross_sim_r2.pkl", "Cross-sim R² (FLAM → TNG)")
held_out_r2    = try_load(RESULTS_DIR / "held_out_r2.pkl", "Held-out R² (overfitting check)")

print(f"\nSummary: {len(flam_samples)}/15 FLAMINGO combos, "
      f"{len(tng_samples)}/{len(BLOCKS)} TNG blocks loaded.")
print(f"best_params: {best_params}")

In [ ]:
# Reuse FLAMINGO models; recompute LG conditioning with the verified convention.
import pickle
CHECKPOINT_PATH = RESULTS_DIR / "flamingo_npe_checkpoint.pkl"
with open(LEGACY_RESULTS_DIR / "flamingo_npe_checkpoint.pkl", "rb") as f:
    legacy_flam = pickle.load(f)
flam_posteriors = legacy_flam["posteriors"]
flam_scalers = legacy_flam["scalers"]
flam_test_scores = legacy_flam["test_scores"]
flam_samples = {}
DATASETS = {"full": flam_df_clean, "same_host": flam_same, "diff_host": flam_diff}
for key, posterior in flam_posteriors.items():
    block = key.split("__")[0]
    flam_samples[key] = sample_lg_point(posterior, flam_scalers[key], BLOCKS[block],
        LG_OBS, h0=H0_BY_SIM["flamingo"], device=device) if block in LG_SAFE_BLOCKS else None
with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump({"posteriors": flam_posteriors, "scalers": flam_scalers,
        "samples": flam_samples, "test_scores": flam_test_scores,
        "best_params": best_params, "run_version": RUN_VERSION,
        "model_description": MODEL_DESCRIPTION, "scenario": "vdM2012 point",
        "model_source": str(LEGACY_RESULTS_DIR / "flamingo_npe_checkpoint.pkl")}, f)
print("Refreshed FLAMINGO LG samples:", CHECKPOINT_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

SPLIT_STYLES = [
    ("full",      "tab:gray",   "--", "full"),
    ("same_host", "tab:orange", "-",  "same-host"),
    ("diff_host", "tab:blue",   "-",  "diff-host (LG-like)"),
]

blocks_done = [b for b in BLOCKS if any(f"{b}__{s}" in flam_samples for s, *_ in SPLIT_STYLES)]

fig, axes = plt.subplots(len(blocks_done), 1, figsize=(7, 3*len(blocks_done)), sharex=True)
if len(blocks_done) == 1:
    axes = [axes]

for ax, block_name in zip(axes, blocks_done):
    for i, (split_name, color, ls, label) in enumerate(SPLIT_STYLES):
        key = f"{block_name}__{split_name}"
        if key not in flam_samples or flam_samples[key] is None:
            continue
        samples = flam_samples[key]
        kde = gaussian_kde(samples)
        xs = np.linspace(samples.min(), samples.max(), 300)
        ax.plot(xs, kde(xs), color=color, ls=ls, label=label)
        med = np.median(samples)
        ax.axvline(med, color=color, ls=":", alpha=0.4, lw=1)

        test_lp = flam_test_scores.get(key)
        tag = f"  (test lp={test_lp:.2f})" if test_lp is not None else ""
        ax.text(0.98, 0.92 - 0.1*i, f"{label}: {med:.3f}{tag}",
                transform=ax.transAxes, ha="right", color=color, fontsize=8)

    ax.set_title(block_name)
    ax.set_ylabel("density")
    ax.legend(loc="upper left", fontsize=8)

axes[-1].set_xlabel(r"$\log_{10}\ M_{200c}$")
plt.tight_layout()
plt.show()

In [ ]:
# TNG: train NPE for all blocks, checkpointed
import pickle, time
from pathlib import Path

TNG_CHECKPOINT_PATH = RESULTS_DIR / "tng_npe_checkpoint.pkl"

tng_posteriors  = {}
tng_scalers     = {}
tng_samples     = {}
tng_test_scores = {}
tng_split_indices = {}

completed_tng = set()
if TNG_CHECKPOINT_PATH.exists():
    with open(TNG_CHECKPOINT_PATH, "rb") as f:
        ckpt = pickle.load(f)
    tng_posteriors.update(ckpt.get("posteriors", {}))
    tng_scalers.update(ckpt.get("scalers", {}))
    tng_samples.update(ckpt.get("samples", {}))
    tng_test_scores.update(ckpt.get("test_scores", {}))
    tng_split_indices.update(ckpt.get("split_indices", {}))
    completed_tng = set(tng_samples.keys())
    if completed_tng:
        print(f"Resuming: {len(completed_tng)} block(s) already done, skipping those.")

density_builder = posterior_nn(
    model="maf",
    hidden_features=best_params["hidden_features"],
    num_transforms=best_params["num_transforms"],
)

n_total = len(BLOCKS)
run_times = []

for block_name, feats in BLOCKS.items():
    if block_name in completed_tng:
        continue

    t0 = time.time()
    print(f"Training TNG: {block_name}  (N={len(tng_df_clean):,})  "
          f"[{len(completed_tng) + len(run_times) + 1}/{n_total}]")

    cols  = feats + [TARGET, "host_i", "host_j"]
    valid = tng_df_clean[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)
    tr, te = group_split(valid, test_size=0.15, seed=42)
    train_valid, test_valid = valid.iloc[tr], valid.iloc[te]
    tng_split_indices[block_name] = {"train": tr, "test": te}

    X_raw = train_valid[feats].values.astype(np.float32)
    y_raw = train_valid[TARGET].values.astype(np.float32)

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)

    theta = torch.tensor(y_raw.reshape(-1, 1), device=device)
    x     = torch.tensor(X_sc, device=device)

    inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
    inference.append_simulations(theta, x)
    density_est = inference.train(
        training_batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"],
        max_num_epochs=200,
        stop_after_epochs=20,
        show_train_summary=False,
    )
    posterior = inference.build_posterior(density_est)

    X_test = scaler.transform(test_valid[feats].values.astype(np.float32))
    theta_test = torch.tensor(test_valid[TARGET].values.astype(np.float32).reshape(-1, 1), device=device)
    x_test     = torch.tensor(X_test, device=device)
    with torch.no_grad():
        test_log_prob = density_est.log_prob(theta_test, condition=x_test).mean().item()

    if block_name in LG_SAFE_BLOCKS:
        lg_vec  = np.array([LG_OBS[f] for f in feats], dtype=np.float32).reshape(1, -1)
        x_obs   = torch.tensor(scaler.transform(lg_to_peculiar(lg_vec, feats, H0_BY_SIM["tng"])), device=device)
        samples = posterior.sample((10_000,), x=x_obs).cpu().numpy().squeeze()
    else:
        samples = None      # simulation-internal block: no LG posterior

    tng_posteriors[block_name]  = posterior
    tng_scalers[block_name]     = scaler
    tng_samples[block_name]     = samples
    tng_test_scores[block_name] = test_log_prob

    elapsed = time.time() - t0
    run_times.append(elapsed)

    if samples is not None:
        print(f"  log M200c: {np.median(samples):.3f} "
              f"[{np.percentile(samples,16):.3f}, {np.percentile(samples,84):.3f}]")
    else:
        print("  LG posterior skipped (not an LG-safe block)")
    print(f"  held-out test log-prob: {test_log_prob:.4f}")
    print(f"  took {elapsed/60:.1f} min")

    with open(TNG_CHECKPOINT_PATH, "wb") as f:
        pickle.dump({
            "posteriors": tng_posteriors,
            "scalers": tng_scalers,
            "samples": tng_samples,
            "test_scores": tng_test_scores,
            "best_params": best_params,
            "split_indices": tng_split_indices, "run_version": RUN_VERSION,
            "model_description": MODEL_DESCRIPTION, "scenario": "vdM2012 point",
        }, f)

print("\nAll TNG blocks done. Saved to", TNG_CHECKPOINT_PATH)

In [ ]:
from scipy.special import rel_entr

def js_divergence(p_samples, q_samples, xs, base=2, bw_method=0.15):
    """Jensen-Shannon divergence between two sample sets, via shared-grid KDE.
    Symmetric, bounded in [0, 1] with base=2. JSD = 0.5*KL(P||M) + 0.5*KL(Q||M),
    M = 0.5*(P+Q).
    """
    p = gaussian_kde(p_samples, bw_method=bw_method)(xs)
    q = gaussian_kde(q_samples, bw_method=bw_method)(xs)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(rel_entr(p, m)) / np.log(base)
    kl_qm = np.sum(rel_entr(q, m)) / np.log(base)
    return 0.5 * kl_pm + 0.5 * kl_qm

xs = np.linspace(11.5, 14.5, 500)

print(f"{'Block':<20} {'JSD (full)':>12} {'JSD (diff-host)':>18} "
      f"{'Median FLAM (full)':>20} {'Median FLAM (diff)':>20} {'Median TNG':>12}")
print("-" * 105)

js_results = {}
for block_name in LG_SAFE_BLOCKS:
    s_flam_full = flam_samples[f"{block_name}__full"]
    s_flam_diff = flam_samples[f"{block_name}__diff_host"]
    s_tng       = tng_samples[block_name]

    jsd_full = js_divergence(s_flam_full, s_tng, xs)
    jsd_diff = js_divergence(s_flam_diff, s_tng, xs)
    js_results[block_name] = {"jsd_full": jsd_full, "jsd_diff_host": jsd_diff}

    print(f"{block_name:<20} {jsd_full:>12.4f} {jsd_diff:>18.4f} "
          f"{np.median(s_flam_full):>20.3f} {np.median(s_flam_diff):>20.3f} "
          f"{np.median(s_tng):>12.3f}")

with open(RESULTS_DIR / "js_divergence_results.pkl", "wb") as f:
    pickle.dump(js_results, f)

In [ ]:
from sklearn.metrics import r2_score

def posterior_point_predictions(posterior, X_raw, scaler, device, n_samples=500):
    X_sc = scaler.transform(X_raw.astype(np.float32))
    x_batch = torch.tensor(X_sc, device=device)
    with torch.no_grad():
        samples = posterior.sample_batched((n_samples,), x=x_batch)
    return samples.mean(dim=0).squeeze(-1).cpu().numpy()


print(f"{'Block':<20} {'R² (FLAM full → TNG)':>22} {'R² (FLAM diff-host → TNG)':>26}")

tng_r2_results = {}
for block_name, feats in BLOCKS.items():
    valid  = tng_df_clean[feats + [TARGET]].dropna()
    X_raw  = valid[feats].values
    y_true = valid[TARGET].values

    r2_scores = {}
    for split_name in ["full", "diff_host"]:
        key = f"{block_name}__{split_name}"
        preds = posterior_point_predictions(
            flam_posteriors[key], X_raw, flam_scalers[key], device
        )
        r2_scores[split_name] = r2_score(y_true, preds)

    tng_r2_results[block_name] = r2_scores
    print(f"{block_name:<20} {r2_scores['full']:>22.3f} {r2_scores['diff_host']:>26.3f}")

with open(RESULTS_DIR / "tng_cross_sim_r2.pkl", "wb") as f:
    pickle.dump(tng_r2_results, f)

In [ ]:
print(f"{'Block':<20} {'R² FLAM full (held-out)':>26} {'R² FLAM diff (held-out)':>26} {'R² TNG (held-out)':>20}")

held_out_r2 = {}
for block_name, feats in BLOCKS.items():
    row = {}
    for split_name, df in [("full", flam_df_clean), ("diff_host", flam_diff)]:
        cols  = feats + [TARGET, "host_i", "host_j"]
        valid = df[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)
        tr, te = group_split(valid, test_size=0.15, seed=42)
        train_valid, test_valid = valid.iloc[tr], valid.iloc[te]

        key = f"{block_name}__{split_name}"
        preds = posterior_point_predictions(
            flam_posteriors[key], test_valid[feats].values, flam_scalers[key], device
        )
        row[f"flam_{split_name}"] = r2_score(test_valid[TARGET].values, preds)

        cols_tng  = feats + [TARGET, "host_i", "host_j"]
        valid_tng = tng_df_clean[cols_tng].dropna(subset=feats + [TARGET]).reset_index(drop=True)
        _, te_tng = group_split(valid_tng, test_size=0.15, seed=42)
        test_valid_tng = valid_tng.iloc[te_tng]
    preds_tng = posterior_point_predictions(
        tng_posteriors[block_name], test_valid_tng[feats].values, tng_scalers[block_name], device
    )
    row["tng"] = r2_score(test_valid_tng[TARGET].values, preds_tng)

    held_out_r2[block_name] = row
    print(f"{block_name:<20} {row['flam_full']:>26.3f} {row['flam_diff_host']:>26.3f} {row['tng']:>20.3f}")

with open(RESULTS_DIR / "held_out_r2.pkl", "wb") as f:
    pickle.dump(held_out_r2, f)